# Mini-TP 7 · Seguridad con SAIF — GraphQL

**Operaciones de Aprendizaje Automático II · CEIA – FIUBA**

Este es **uno de los seis notebooks guiados** del Mini-TP 7. Cada uno asegura **una capa**
de las que construimos en el bimestre. **Elige este si tu modelo se expone por GraphQL**
(o simplemente para practicar esta estrategia). Con completar **uno** alcanza para la entrega.

### SAIF en esta capa
**Riesgo:** DoS por consultas profundas/costosas y **fuga del esquema por introspección**.  
**Controles:** límite de profundidad/costo · introspección apagada · autorización por campo.  
**Elementos SAIF:** automatizar defensas · bases sólidas.

### Qué vas a hacer
1. Ver la **versión vulnerable** y por qué es un riesgo.
2. Aplicar el **control principal** (guiado, ya resuelto como ejemplo).
3. Completar los **`# TODO`** sobre tu propio modelo.
4. Escribir una **reflexión** (amenaza mitigada ↔ elemento de SAIF).

> Corre de punta a punta con **uv**, sin servicios externos.


### Dependencias
```bash
uv add strawberry-graphql
```

## 1. Versión VULNERABLE — introspección abierta y sin límite de profundidad

In [ ]:
import strawberry

@strawberry.type
class Prediccion:
    clase: int
    confianza: float

@strawberry.type
class QueryV:
    @strawberry.field
    def predecir(self, x: float) -> Prediccion:
        return Prediccion(clase=int(x > 0.5), confianza=0.9)

schema_v = strawberry.Schema(QueryV)   # introspección ON, sin límite de profundidad

# Un atacante mapea TODO el esquema con introspección:
r = schema_v.execute_sync("{ __schema { types { name } } }")
print("Tipos expuestos por introspección:", len(r.data["__schema"]["types"]),
      "-> el atacante conoce toda la API")

## 2. Control principal (GUIADO) — auth por rol + sin introspección + límite de profundidad

In [ ]:
from strawberry.extensions import QueryDepthLimiter, AddValidationRules
from graphql import NoSchemaIntrospectionCustomRule

@strawberry.type
class Query:
    @strawberry.field
    def predecir(self, info: strawberry.Info, x: float) -> Prediccion:
        # autorización por campo: el rol viaja en el context
        if info.context.get("rol") not in ("analista", "admin"):
            raise Exception("no autorizado")
        return Prediccion(clase=int(x > 0.5), confianza=0.9)

# factories (lambda) para construir una extensión fresca por request
schema = strawberry.Schema(
    Query,
    extensions=[lambda: QueryDepthLimiter(max_depth=5),
                lambda: AddValidationRules([NoSchemaIntrospectionCustomRule])],
)

sin_rol = schema.execute_sync("{ predecir(x:0.8){clase} }", context_value={})
con_rol = schema.execute_sync("{ predecir(x:0.8){clase confianza} }", context_value={"rol": "analista"})
intro   = schema.execute_sync("{ __schema { types { name } } }", context_value={"rol": "analista"})
print("sin rol      :", sin_rol.errors[0].message)
print("con rol      :", con_rol.data)
print("introspección:", "bloqueada" if intro.errors else "ABIERTA")

## 3. Tu entrega — completa los `# TODO`

In [ ]:
# TODO 1: adapta 'predecir' para que reciba las features REALES de tu modelo
#   (p.ej. una lista) y devuelva la predicción de tu modelo entrenado.
# TODO 2: agrega un límite de COSTO además del de profundidad
#   (rechazá listas sin paginar o pide un argumento 'limit' con tope).
# TODO 3: autorización por campo más fina: que un campo sensible
#   (p.ej. 'confianza' o metadatos del modelo) requiera rol 'admin'.
print("Completa los TODO sobre tu propio modelo.")

## 4. Reflexión
1. ¿Por qué apagar la introspección en producción reduce el riesgo?
2. ¿Qué ataque frena el límite de profundidad/costo?
3. ¿Qué elemento de SAIF cubre tu control principal?

*(Escribe tu respuesta aquí.)*